In [1]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

bronze_path = "Files/nyc_taxi/yellow"
silver_path = "abfss://440b1492-88fd-44d5-bac9-a81607488e15@onelake.dfs.fabric.microsoft.com/5d1d0a76-8429-4af1-967a-32514d24592c/Tables/dbo/nyc_taxi_trips"

min_duration = 1
max_duration = 180
max_distance = 100
max_speed = 80
max_fare = 1000
bad_zones = [264, 265]

In [3]:
files = []
for year_folder in notebookutils.fs.ls(bronze_path):
    for month_folder in notebookutils.fs.ls(year_folder.path):
        for file in notebookutils.fs.ls(month_folder.path):
            if file.name.endswith(".parquet"):
                files.append(file.path)

print(len(files))

In [5]:
def read_taxi_file(path):
    df = spark.read.parquet(path)
    df = df.toDF(*[c.lower() for c in df.columns])

    if "cbd_congestion_fee" not in df.columns:
        df = df.withColumn("cbd_congestion_fee", F.lit(None))

    year = int(path.split("year=")[1][:4])
    month = int(path.split("month=")[1][:2])

    return df.select(
        F.col("vendorid").cast("int").alias("vendor_id"),
        F.col("tpep_pickup_datetime").cast("timestamp").alias("pickup_datetime"),
        F.col("tpep_dropoff_datetime").cast("timestamp").alias("dropoff_datetime"),
        F.col("passenger_count").cast("int"),
        F.col("trip_distance").cast("double"),
        F.col("ratecodeid").cast("int").alias("rate_code_id"),
        F.col("store_and_fwd_flag"),
        F.col("pulocationid").cast("int").alias("pu_location_id"),
        F.col("dolocationid").cast("int").alias("do_location_id"),
        F.col("payment_type").cast("int"),
        F.col("fare_amount").cast("double"),
        F.col("extra").cast("double"),
        F.col("mta_tax").cast("double"),
        F.col("tip_amount").cast("double"),
        F.col("tolls_amount").cast("double"),
        F.col("improvement_surcharge").cast("double"),
        F.col("total_amount").cast("double"),
        F.col("congestion_surcharge").cast("double"),
        F.col("airport_fee").cast("double"),
        F.col("cbd_congestion_fee").cast("double"),
        F.lit(year).alias("year"),
        F.lit(month).alias("month"),
    )


df = read_taxi_file(files[0])
for path in files[1:]:
    df = df.unionByName(read_taxi_file(path))

df.printSchema()

In [8]:
clean = (
    df
    .withColumn("trip_duration_min", F.round((F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")) / 60, 2))
    .withColumn("passenger_count", F.when(F.col("passenger_count") == 0, None).otherwise(F.col("passenger_count")))
    .withColumn("rate_code_id", F.when(F.col("rate_code_id") == 99, None).otherwise(F.col("rate_code_id")))
    .filter(F.year("pickup_datetime") == F.col("year"))
    .filter(F.month("pickup_datetime") == F.col("month"))
    .filter(F.col("trip_duration_min") >= min_duration)
    .filter(F.col("trip_duration_min") <= max_duration)
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("trip_distance") <= max_distance)
    .filter(F.col("trip_distance") <= F.col("trip_duration_min") / 60 * max_speed)
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("fare_amount") <= max_fare)
    .filter(F.col("total_amount") > 0)
    .filter(~F.col("pu_location_id").isin(bad_zones))
    .filter(~F.col("do_location_id").isin(bad_zones))
    .dropDuplicates()
)

In [9]:
silver = (
    clean
    .withColumn("pickup_date", F.to_date("pickup_datetime"))
    .withColumn("pickup_hour", F.hour("pickup_datetime"))
    .withColumn("day_of_week", F.dayofweek("pickup_datetime"))
    .withColumn("is_weekend", F.dayofweek("pickup_datetime").isin(1, 7))
)

In [10]:
silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").partitionBy("year", "month").save(silver_path)

In [12]:
result = spark.read.format("delta").load(silver_path)

print(df.count())
print('result', result.count())
display(result.groupBy("year", "month").count().orderBy("year", "month"))